# EDA Site PLTS untuk Machine Learning

Exploratory Data Analysis untuk **site daily** (`mart.mart_site_performance_daily`), **device 5min** (meter, sensor, inverter) per site, dan atribut dari `dim_assets`. **Exclude: Samator & Klinik** (untuk device 5min dan opsional untuk site daily). Checklist: shape, missing, univariate, multivariate, time series, outlier, ML readiness.

## 1. Data loading & shape

Load dari PostgreSQL. Env dari **eda/.env**: POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD (atau PGHOST, PGPORT, …). Fallback CSV jika tidak ada koneksi.

In [1]:
import os
import pandas as pd
import numpy as np

try:
    from dotenv import load_dotenv
    from pathlib import Path
    _dir = Path().resolve()
    if (_dir / "eda" / ".env").exists():
        load_dotenv(_dir / "eda" / ".env")
    else:
        load_dotenv(_dir / ".env")
except ImportError:
    pass

# Try PostgreSQL first (eda/.env: POSTGRES_* or PGHOST/...)
try:
    import psycopg2
    from sqlalchemy import create_engine
    
    host = os.environ.get('POSTGRES_HOST') or os.environ.get('PGHOST', '10.101.4.88')
    port = os.environ.get('POSTGRES_PORT') or os.environ.get('PGPORT', '5432')
    user = os.environ.get('POSTGRES_USER') or os.environ.get('PGUSER', 'juice')
    password = os.environ.get('POSTGRES_PASSWORD') or os.environ.get('PGPASSWORD', '')
    dbname = os.environ.get('POSTGRES_DB') or os.environ.get('PGDATABASE', 'MMSR')
    
    if password:
        conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
        engine = create_engine(conn_str)
        # Exclude Samator & Klinik (konsisten dengan device 5min)
        df = pd.read_sql(
            """SELECT * FROM mart.mart_site_performance_daily
               WHERE site_name NOT ILIKE '%Samator%' AND site_name NOT ILIKE '%Klinik%'
               ORDER BY date_key, site_id""",
            engine
        )
        print("Loaded from PostgreSQL:", df.shape)
    else:
        raise ValueError("Set POSTGRES_PASSWORD (or PGPASSWORD) in eda/.env or load from CSV")
except Exception as e:
    csv_path = 'mart_site_performance_daily.csv'
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['date_key'] = pd.to_datetime(df['date_key']).dt.date
        print("Loaded from CSV:", df.shape)
    else:
        raise SystemExit("No DB connection and no CSV. Set POSTGRES_* in eda/.env or export to CSV.")

print("Rows:", len(df), "Columns:", len(df.columns))
print(df.dtypes.head(15))

SystemExit: No DB connection and no CSV. Set POSTGRES_* in eda/.env or export to CSV.

c:\Users\Administrator\Documents\Code\MMSR API - Server MA\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Ensure date_key is datetime-like for time series
df['date_key'] = pd.to_datetime(df['date_key'])

# Missing % per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'null_count': missing, 'null_pct': missing_pct})
print(missing_df[missing_df['null_count'] > 0].sort_values('null_count', ascending=False))

## 2. Univariate

Distribusi numerik (energy, GHI, PR, availability); value counts site/system; coverage waktu.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

num_cols = ['daily_energy_mwh', 'daily_ghi_kwh_m2', 'availability_percent', 'pr_ghi_actual', 'pr_poa_actual']
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
if len(num_cols) < 6:
    for j in range(len(num_cols), 6):
        axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical: site_id, system
print("Value counts system:")
print(df['system'].value_counts())
print("\nValue counts site_id (top 10):")
print(df['site_id'].value_counts().head(10))

# Date coverage per site
coverage = df.groupby('site_id')['date_key'].agg(['min', 'max', 'count'])
coverage.columns = ['first_date', 'last_date', 'days']
coverage = coverage.sort_values('days', ascending=False)
print("\nDate coverage per site:")
print(coverage)

## 3. Multivariate

Heatmap korelasi; scatter energy vs GHI, PR vs availability; boxplot per site/bulan.

In [ ]:
feat_cols = ['daily_energy_mwh', 'daily_ghi_kwh_m2', 'daily_poa_weighted_kwh_m2', 'availability_percent', 'pr_ghi_actual', 'pr_poa_actual', 'energy_target_mwh']
feat_cols = [c for c in feat_cols if c in df.columns]
corr = df[feat_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation matrix (numeric features)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df['daily_ghi_kwh_m2'], df['daily_energy_mwh'], alpha=0.3, s=10)
axes[0].set_xlabel('daily_ghi_kwh_m2')
axes[0].set_ylabel('daily_energy_mwh')
axes[0].set_title('Energy vs GHI')
axes[1].scatter(df['availability_percent'], df['pr_ghi_actual'], alpha=0.3, s=10)
axes[1].set_xlabel('availability_percent')
axes[1].set_ylabel('pr_ghi_actual')
axes[1].set_title('PR GHI vs Availability')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot daily_energy_mwh by site (top 8 by count)
top_sites = df['site_id'].value_counts().head(8).index.tolist()
df_top = df[df['site_id'].isin(top_sites)]
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_top, x='site_id', y='daily_energy_mwh', order=top_sites)
plt.xticks(rotation=45, ha='right')
plt.title('Daily energy (MWh) by site (top 8)')
plt.tight_layout()
plt.show()

## 4. Time series

Rata-rata harian/bulanan energy dan PR; seasonality; gap/missing days.

In [ ]:
# Daily average energy (all sites) over time
daily_agg = df.groupby('date_key').agg({'daily_energy_mwh': 'mean', 'pr_ghi_actual': 'mean'}).reset_index()
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(daily_agg['date_key'], daily_agg['daily_energy_mwh'], alpha=0.8)
axes[0].set_ylabel('Avg daily_energy_mwh')
axes[0].set_title('Daily average energy (all sites)')
axes[1].plot(daily_agg['date_key'], daily_agg['pr_ghi_actual'], alpha=0.8)
axes[1].set_ylabel('Avg pr_ghi_actual')
axes[1].set_xlabel('date_key')
axes[1].set_title('Daily average PR GHI (all sites)')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly seasonality: average energy by month
df['month'] = df['date_key'].dt.month
monthly = df.groupby('month')['daily_energy_mwh'].mean()
plt.figure(figsize=(10, 4))
monthly.plot(kind='bar')
plt.xlabel('Month')
plt.ylabel('Avg daily_energy_mwh')
plt.title('Seasonality: average daily energy by month')
plt.tight_layout()
plt.show()

## 5. Outlier & data quality

Z-score / IQR; hari dengan energy/PR nol atau negatif; pengaruh is_issue_date.

In [ ]:
# Zero / negative checks
print("Rows with daily_energy_mwh <= 0:", (df['daily_energy_mwh'] <= 0).sum())
print("Rows with pr_ghi_actual <= 0 or > 1.5:", ((df['pr_ghi_actual'] <= 0) | (df['pr_ghi_actual'] > 1.5)).sum())

# IQR outlier (energy)
Q1 = df['daily_energy_mwh'].quantile(0.25)
Q3 = df['daily_energy_mwh'].quantile(0.75)
IQR = Q3 - Q1
out_energy = (df['daily_energy_mwh'] < (Q1 - 1.5*IQR)) | (df['daily_energy_mwh'] > (Q3 + 1.5*IQR))
print("Outliers (IQR 1.5x) daily_energy_mwh:", out_energy.sum())

# is_issue_date impact
if 'is_issue_date' in df.columns:
    print("\nMean daily_energy_mwh by is_issue_date:")
    print(df.groupby('is_issue_date')['daily_energy_mwh'].agg(['mean', 'count']))

## 6. Kesiapan ML

Definisi target; feature kandidat; train/val/test split (time-based); handling missing dan encoding.

In [ ]:
print("=== ML Readiness ===")
print("Suggested targets: daily_energy_mwh, pr_ghi_actual, pr_poa_actual (or energy_target_mwh for subset)")
print("Feature candidates: date_key (month, day_of_week), daily_ghi_kwh_m2, daily_poa_weighted_kwh_m2, availability_percent, actual_capacity_kw, system, site_id (encoded), is_issue_date")
print("\nTime-based split: e.g. train < 2025-10, val 2025-10 to 2025-11, test >= 2025-12")

# Example: filter sites with enough history (e.g. >= 90 days)
site_days = df.groupby('site_id')['date_key'].nunique()
sites_90 = site_days[site_days >= 90].index.tolist()
print(f"\nSites with >= 90 days: {len(sites_90)} (of {df['site_id'].nunique()})")
print("Missing: impute GHI/POA per site (e.g. forward fill) or drop rows with null target; encode site_id (one-hot or embedding), system (one-hot).")

## 7. Device 5min EDA (Meter, Sensor, Inverter) — Exclude Samator & Klinik

Load **daily aggregates** dari 5min data (query `eda_device_5min_daily_aggregates.sql`) agar ukuran manageable. Filter: `site_name NOT ILIKE '%Samator%' AND NOT ILIKE '%Klinik%'`.

In [ ]:
# Load daily aggregates (excl. Samator & Klinik). Butuh koneksi DB.
EXCLUDE_FILTER = "site_name NOT ILIKE '%Samator%' AND site_name NOT ILIKE '%Klinik%'"

def load_device_daily_agg(engine, table, device_type):
    q = f"""
    SELECT date_key, site_name, system, asset_id, metric_name, metric_unit,
           COUNT(*) AS n_points, ROUND(AVG(metric_value)::numeric, 6) AS avg_value,
           MIN(metric_value) AS min_value, MAX(metric_value) AS max_value
    FROM mart.{table}
    WHERE {EXCLUDE_FILTER} AND metric_value IS NOT NULL
    GROUP BY date_key, site_name, system, asset_id, metric_name, metric_unit
    """
    df_agg = pd.read_sql(q, engine)
    df_agg['date_key'] = pd.to_datetime(df_agg['date_key'])
    df_agg['device_type'] = device_type
    return df_agg

if 'engine' in dir() and engine is not None:
    df_meter = load_device_daily_agg(engine, 'mart_meter_performance_5min', 'meter')
    df_sensor = load_device_daily_agg(engine, 'mart_sensor_measurements_5min', 'sensor')
    df_inv = load_device_daily_agg(engine, 'mart_inverter_performance_5min', 'inverter')
    print("Meter daily agg:", df_meter.shape)
    print("Sensor daily agg:", df_sensor.shape)
    print("Inverter daily agg:", df_inv.shape)
else:
    df_meter = df_sensor = df_inv = None
    print("Set engine (run site daily load cell) then re-run this cell.")

In [ ]:
# Univariate: metric_name value counts and avg_value distribution per device
if df_meter is not None:
    for name, d in [('Meter', df_meter), ('Sensor', df_sensor), ('Inverter', df_inv)]:
        print(f"--- {name} ---")
        print(d['metric_name'].value_counts().head(10))
        print()
    # Distribution of avg_value for key metrics (sample)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (name, d) in zip(axes, [('Meter', df_meter), ('Sensor', df_sensor), ('Inverter', df_inv)]):
        top_metric = d['metric_name'].value_counts().index[0]
        sub = d[d['metric_name'] == top_metric]['avg_value'].dropna()
        sub = sub[(sub > sub.quantile(0.01)) & (sub < sub.quantile(0.99))]
        ax.hist(sub, bins=50, edgecolor='black', alpha=0.7)
        ax.set_title(f"{name}: {top_metric}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Device 5min: rows per site (daily agg = one row per date/site/asset/metric)
if df_meter is not None:
    for name, d in [('Meter', df_meter), ('Sensor', df_sensor), ('Inverter', df_inv)]:
        site_counts = d.groupby('site_name').size().sort_values(ascending=False)
        print(f"{name} — rows per site (daily agg):")
        print(site_counts.head(10))
        print()
    # Time series: avg metric value over time (e.g. inverter inv_active_power)
    if 'inv_active_power' in df_inv['metric_name'].values:
        inv_power = df_inv[df_inv['metric_name'] == 'inv_active_power'].groupby('date_key')['avg_value'].mean()
        plt.figure(figsize=(12, 3))
        plt.plot(inv_power.index, inv_power.values, alpha=0.8)
        plt.ylabel('Avg inv_active_power (daily agg)')
        plt.title('Inverter active power (daily avg, all sites) — excl. Samator/Klinik')
        plt.tight_layout()
        plt.show()